In [ ]:
import sys
#!pip install polars --target ./my_custom_packages
#sys.path.append("./my_custom_packages") #
custom_path = "/stor/home/he4249/orange/my_custom_packages" # lifetimes, numpy<2.0.0, tslearn
sys.path.insert(0, custom_path)

import polars as pl
import pandas as pd
import numpy as np

# Deprecated aliases back into NumPy so older libraries don't crash

np.long = int
np.ulong = int  
np.float = float # for lifetimes
np.bool = bool
np.object = object
np.int = int
np.unicode = str
np.str = str
np.complex = complex

import seaborn as sns
import matplotlib.pyplot as plt

from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data, calibration_and_holdout_data
from lifetimes.plotting import plot_frequency_recency_matrix, plot_period_transactions

from sklearn.metrics import mean_squared_error

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

from tslearn.clustering import TimeSeriesKMeans
from tslearn.utils import to_time_series_dataset

from joblib import parallel_backend


pl.Config.set_tbl_rows(50)
pl.Config(tbl_cols = 50)

In [ ]:
df_merged = pl.read_parquet("df_merged.parquet")
display(df_merged.head())
display(df_merged.shape)

In [ ]:
df_tsa = df_merged.sort("date").with_columns(
    (pl.col("date") - pl.col("date").shift(1).over("msisdn")).dt.total_days().alias("tblp_last"),
    (pl.col("date").shift(-1).over("msisdn") - pl.col("date")).dt.total_days().alias("tbnp_next")
)

df_tsa.head()

In [ ]:
display(df_tsa["tblp_last", "tbnp_next"].describe())

display(df_tsa.group_by("msisdn").agg(
    pl.col("tblp_last").mean().alias("avg_user_tbp"),
    pl.col("tblp_last").median().alias("median_user_tbp")
)["avg_user_tbp", "median_user_tbp"].describe())

df_tsa_pd = df_tsa.to_pandas()
# 500k users bought once and stopped. LOOK INTO IT.

In [ ]:
%%time
# 10% of data
df_sample = df_tsa_pd.sample(frac= 0.10, random_state = 4)

max_date = pd.to_datetime(df_sample["date"]).max()
split_date = max_date - pd.Timedelta(days = 20)

cal_holdout = calibration_and_holdout_data(
    df_sample, 
    customer_id_col = "msisdn", 
    datetime_col = "date",
    calibration_period_end = split_date,
    observation_period_end = max_date,
    monetary_value_col = "CA"
)

penalizers_to_test = [0.001, 0.01, 0.1, 1.0] # convergence error 0.0
best_penalizer = 0.0
best_mse = float("inf")

for pen in penalizers_to_test:
    bgf_tune = BetaGeoFitter(penalizer_coef = pen)
    
    bgf_tune.fit(
        cal_holdout["frequency_cal"], 
        cal_holdout["recency_cal"], 
        cal_holdout["T_cal"]
    )
    
    predicted_purchases = bgf_tune.predict(
        30, 
        cal_holdout["frequency_cal"], 
        cal_holdout["recency_cal"], 
        cal_holdout["T_cal"]
    )

    if predicted_purchases.isna().any():
        continue

    mse = mean_squared_error(cal_holdout["frequency_holdout"], predicted_purchases)
    
    if mse < best_mse:
        best_mse = mse
        best_penalizer = pen

display(best_penalizer)


In [ ]:
%%time 

best_penalizer = 0.01

rfm = summary_data_from_transaction_data(
    df_tsa_pd,
    customer_id_col = "msisdn",
    datetime_col = "date",
    monetary_value_col = "CA"
)
# T: Age, first purchase to end of period

bgf = BetaGeoFitter(penalizer_coef = best_penalizer)
bgf.fit(rfm["frequency"], rfm["recency"], rfm["T"])
display(bgf.summary)

t = 30 # Next 30 days
rfm["predicted_purchases_for_next_t_days"] = bgf.conditional_expected_number_of_purchases_up_to_time(
    t,
    rfm["frequency"],
    rfm["recency"],
    rfm["T"]
)

rfm["prob_alive"] = bgf.conditional_probability_alive(
    rfm["frequency"],
    rfm["recency"],
    rfm["T"]
)

display(rfm.head())

# expected purchases by recency and frequency
plot_frequency_recency_matrix(bgf)
# check model fit against actual data
plot_period_transactions(bgf)


In [ ]:
display(rfm["prob_alive"].describe())
plt.hist(rfm["prob_alive"], bins = 50, color = "blue")
plt.title("Distribution of alive probability calculated by btyd")
plt.xlabel("Proba that user is alive")
plt.ylabel("Number of Customers")
plt.show()

display(sum(rfm["prob_alive"] < 0.5))
display(sum(rfm["prob_alive"] < 0.2))
display(sum(rfm["prob_alive"] < 0.1))

In [ ]:
# Positve, gamma function requirement
g_summary = rfm[(rfm["frequency"] > 0) & (rfm["monetary_value"] > 0)]

ggf = GammaGammaFitter()
ggf.fit(g_summary["frequency"], g_summary["monetary_value"])

rfm["expected_average_monetary_value"] = ggf.conditional_expected_average_profit(
    rfm["frequency"], rfm["monetary_value"]
)

rfm["clv_12months"] = ggf.customer_lifetime_value(
    bgf,
    rfm["frequency"],
    rfm["recency"],
    rfm["T"],
    rfm["monetary_value"],
    time = 12,
    discount_rate  = 0.01 # Expected loss of value per month, about 13% annually: Professors Peter Fader and Bruce Hardie who developed the BG/NBD and Gamma-Gamma models used d = 0.01
            
    
)
rfm.head()


In [ ]:
%%time

user_plans = df_tsa.filter(pl.col("Nom du forfait").is_not_null()).sort(["msisdn", "date"]).group_by("msisdn").agg(
    pl.col("Nom du forfait").alias("purchase_sequence")
)["purchase_sequence"].to_list()

te = TransactionEncoder()
te_plans = te.fit_transform(user_plans)

df_plans_pd = pd.DataFrame(te_plans, columns = te.columns_)

# min_support: must appear in 1% of users
frequent_plans = fpgrowth(df_plans_pd, min_support = 0.01, use_colnames = True)

# min_threshold: min 20% confidence
# confidence: of all the people who bought A what percentage of them also bought B
rules = association_rules(frequent_plans, metric = "confidence", min_threshold = 0.20)

# 1 product leading to 1 product
nbo_rules = rules[
    (rules["antecedents"].apply(lambda x: len(x) == 1)) & 
    (rules["consequents"].apply(lambda x: len(x) == 1))
]

# Confidence: confidence of 0.65 means 65% of people who bought the antecedent went on to buy the consequent
# Lift: a lift of 3.5 means a user is 3.5 times more likely to buy Akama Up because they previously bought Be 500 New than a random user
nbo_rules = nbo_rules.sort_values(["antecedents", "lift"], ascending = [True, False])
best_rules = nbo_rules.drop_duplicates(subset = ["antecedents"], keep = "first")
best_rules["product_bought"] = best_rules["antecedents"].apply(lambda x: list(x)[0])
best_rules["likely_following_product"] = best_rules["consequents"].apply(lambda x: list(x)[0])

display(best_rules)

In [ ]:
# last plan purchased
last_plans = df_tsa.filter(pl.col("Nom du forfait").is_not_null()).sort("date").group_by("msisdn").agg(
    pl.col("Nom du forfait").last().alias("last_plan_bought")
    ).to_pandas()

customer_profile = rfm.merge(last_plans, on = "msisdn", how = "left")

# Map next best offer
nbo_map = best_rules.set_index("product_bought")["likely_following_product"].to_dict()
customer_profile["next_best_offer"] = customer_profile["last_plan_bought"].map(nbo_map)

# Fill missing with personal most popular package
top_default_plan = df_merged["Nom du forfait"].mode()[0]
customer_profile["next_best_offer"] = customer_profile["next_best_offer"].fillna(top_default_plan)

display(customer_profile.head())


In [ ]:
def person_bucket(row):
    # Active and high predicted transactions
    if row["prob_alive"] > 0.7 and row["predicted_purchases_for_next_t_days"] > 6.0:
        return "Upsell by next best offer"
    # Frequent but at risk of leaving
    elif 0.3 <= row["prob_alive"] <= 0.7:
        return "Reengagement by discount"
    # Churning
    elif row["prob_alive"] < 0.3:
        return "Win back"
    else:
        return "Standard"

customer_profile["marketing_segment"] = customer_profile.apply(person_bucket, axis = 1)
display(customer_profile.head())
display(customer_profile["marketing_segment"].value_counts())

In [ ]:
# One time buyers in three months, why?

max_date = df_tsa["date"].max()

user_summary = df_tsa.sort("date").group_by("msisdn").agg(
    pl.len().alias("purchase_count"),
    pl.col("date").min().alias("first_date"),
    pl.col("Nom du forfait").first().alias("plan_bought"),
    pl.col("CA").sum().alias("total_spent")
)

one_timers = user_summary.filter(pl.col("purchase_count") == 1)
display(f"n one timers: {len(one_timers)}")

plan_tendencies = (
    user_summary.group_by("plan_bought")
    .agg(
        pl.len().alias("total_buyers"),
        (pl.col("purchase_count") == 1).sum().alias("one_time_buyers"),
        # Avg days since purchase for one-timers (tells you if they are lost vs. brand new)
        (max_date - pl.col("first_date")).filter(pl.col("purchase_count") == 1).dt.total_days().mean().round(1).alias("avg_days_ago")
    )
    .with_columns(
        (pl.col("one_time_buyers") / pl.col("total_buyers") * 100).round(2).alias("onetimers_rate")
    )
    .filter(pl.col("total_buyers") > 10000)  # low volume noise
    .sort("onetimers_rate", descending = True)
)

display(plan_tendencies)

In [ ]:
segment_financials = customer_profile.groupby("marketing_segment").agg(
    user_count = ("msisdn", "count"),
    total_clv_12months = ("clv_12months", "sum"),
    avg_clv_12months = ("clv_12months", "mean")
).reset_index()

segment_financials["total_clv_billion"] = (segment_financials["total_clv_12months"] / 1_000_000_000).round(2)
segment_financials = segment_financials.sort_values("total_clv_billion", ascending = False)

display(segment_financials[["marketing_segment", "user_count", "total_clv_billion", "avg_clv_12months"]])

In [ ]:
df_datatech_region = df_tsa.sort("date").group_by("msisdn").agg(
    pl.col("region_cleaned").last(),
    pl.col("Typologie").last(),
    pl.col("Max_RAT").last()
).to_pandas()

customer_profile_full = customer_profile.merge(df_datatech_region, on = "msisdn", how = "left")

more_user_info = customer_profile_full.groupby(["Max_RAT", "Typologie"]).agg(
    n_user = ("msisdn", "count"),
    avg_prob_alive = ("prob_alive", "mean"),
    avg_clv_12months = ("clv_12months", "mean")
).reset_index().sort_values("avg_clv_12months", ascending = False)

display(more_user_info)

# Which regions have the highest concentration of churning users
region_churn = customer_profile_full[customer_profile_full["marketing_segment"] == "Win back"]
region_churn_stats = region_churn.groupby("region_cleaned").size().sort_values(ascending=False)
display(region_churn_stats / len(region_churn) )

In [ ]:
legacy_vs_new = one_timers.filter(
    pl.col("plan_bought").is_in(["be 500", "be 500 new"])
).to_pandas()

for plan in ["be 500", "be 500 new"]:
    subset = legacy_vs_new[legacy_vs_new["plan_bought"] == plan]
    subset["first_date"].dt.date.value_counts().sort_index().plot(label = plan)

plt.title("Timeline of one timer with the old and new Be plan")
plt.ylabel("Number of one timers")
plt.xlabel("Date")
plt.legend()
plt.show()

In [ ]:
plan_prices = df_tsa_pd.drop_duplicates(subset = ["Nom du forfait"])[["Nom du forfait", "CA"]].set_index("Nom du forfait")["CA"].to_dict()

upsell_segment = customer_profile[customer_profile["marketing_segment"] == "Upsell by next best offer"]
upsell_segment["last_plan_price"] = upsell_segment["last_plan_bought"].map(plan_prices)
upsell_segment["nbo_price"] = upsell_segment["next_best_offer"].map(plan_prices)

# price difference
upsell_segment["price_diff"] = upsell_segment["nbo_price"] - upsell_segment["last_plan_price"]

def categorize_upsell(diff):
    if diff > 0:
        return "+ revenue"
    elif diff == 0:
        return "Retention"
    else:
        return "- revenue"

upsell_segment["nbo_revenue"] = upsell_segment["price_diff"].apply(categorize_upsell)

display(upsell_segment["nbo_revenue"].value_counts())

In [ ]:
best_rules["antecedent_price"] = best_rules["product_bought"].map(plan_prices)
best_rules["consequent_price"] = best_rules["likely_following_product"].map(plan_prices)

profitable_rules = best_rules[best_rules["consequent_price"] >= best_rules["antecedent_price"]]

profitable_nbo_map = profitable_rules.set_index("product_bought")["likely_following_product"].to_dict()

# Remap NBO for customer profile
customer_profile["next_best_offer_safe"] = customer_profile["last_plan_bought"].map(profitable_nbo_map)

# default to the overall top plan
customer_profile["next_best_offer_safe"] = customer_profile["next_best_offer_safe"].fillna(top_default_plan)

upsell_segment_2 = customer_profile[customer_profile["marketing_segment"] == "Upsell by next best offer"]
upsell_segment_2["last_plan_price"] = upsell_segment_2["last_plan_bought"].map(plan_prices)
upsell_segment_2["nbo_safe_price"] = upsell_segment_2["next_best_offer_safe"].map(plan_prices)
upsell_segment_2["price_diff_safe"] = upsell_segment_2["nbo_safe_price"] - upsell_segment_2["last_plan_price"]

upsell_segment_2["nbo_revenue_safe"] = upsell_segment_2["price_diff_safe"].apply(categorize_upsell)

display(upsell_segment_2["nbo_revenue_safe"].value_counts())

In [ ]:
legacy_vs_new = one_timers.filter(
    pl.col("plan_bought").is_in(["ao tsara"])
).to_pandas()

for plan in ["ao tsara"]:
    subset = legacy_vs_new[legacy_vs_new["plan_bought"] == plan]
    subset["first_date"].dt.date.value_counts().sort_index().plot(label = plan)

plt.title("Timeline of one timer with the old and new ao tsara")
plt.ylabel("Number of one timers")
plt.xlabel("Date")
plt.legend()
plt.show()

In [ ]:
df_ts = df_tsa.filter(df_tsa["msisdn"].is_in(rfm[rfm["prob_alive"] < 0.1].reset_index()["msisdn"]))
display(df_ts.head())
display(df_ts.shape)
display(df_ts.dtypes)
df_ts.write_parquet("df_ts.parquet")

In [ ]:
df_ts = pl.read_parquet("df_ts.parquet")

In [ ]:
df_ts_ready = df_ts.group_by("msisdn").agg(pl.col("tblp_last").cast(pl.Float64).tail(50))
display(df_ts_ready.head())
display(df_ts_ready.shape)
display(df_ts_ready.dtypes)

dt_ts = to_time_series_dataset(df_ts_ready["tblp_last"].to_list())

In [ ]:
%%time

k_range = range(2, 15)
inertias = []

for k in k_range:
    model = TimeSeriesKMeans(n_clusters = k, metric = "dtw", n_jobs = -1, random_state = 4)
    model.fit(dt_ts)
    inertias.append(model.inertia_)
    print(f"k = {k}")

plt.plot(k_range, inertias, marker = "o")
plt.show()

In [ ]:
model = TimeSeriesKMeans(
    n_clusters = 3, 
    metric = "dtw", 
    metric_params = {"global_constraint": "sakoe_chiba", 
                     "sakoe_chiba_radius": 50.0, # all transactions
                     "itakura_max_slope": float("inf")}, # unimportant, we're doing sakoe_chiba
    random_state = 4, 
    n_jobs = 8, # 8 out of 112 threads or 56 phyisical cores
    ) # goes to threadingbackend
    # verbose = 50

with parallel_backend("loky", n_jobs = 8):
    model.fit(dt_ts)


In [ ]:
model = TimeSeriesKMeans(
    n_clusters=3,
    metric="dtw",
    metric_params={
        "global_constraint": "sakoe_chiba",
        "sakoe_chiba_radius": 50,
    },
    n_jobs=8,
    max_iter=1,
    max_iter_barycenter=1,
    n_init=1,
    random_state=4,
    verbose=True,
)

model.fit(dt_ts)